# Reconstrucción limpia del modelo RF124 leaf16

## Alcance y trazabilidad

- Matriz maestra: `notebooks/data/interim/tabular/exp38_c_matriz_maestra.csv`.
- Folds espaciales: `notebooks/EXP24_predicciones_oof.csv`.
- Parcela con rendimiento conocido: 138.
- Parcelas sin rendimiento observado: 59.
- Variables RF124: 124 (51 satelitales, 72 climáticas y `AREA_HA`).
- Modelo: `RandomForestRegressor(n_estimators=500, min_samples_leaf=16, max_features=1.0, random_state=2026)`.

**Importante:** este notebook reconstruye el modelado a partir de la matriz maestra ya generada por EXP38-C. No vuelve a generar desde cero las siete tablas fuente (`planet_features`, `chirps_ancho`, `tmax_ancho`, `tmin_ancho`, `tmedia_ancho`, `clima_resumen` y `topo_features`). Para reproducir también la extracción y transformación de esas fuentes, esos pasos upstream y sus archivos de entrada deben conservarse en el proyecto.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

print("scikit-learn:", sklearn.__version__)
print("Directorio de trabajo:", Path.cwd().resolve())

In [ ]:
# Localizar la raíz del proyecto sin depender de una ruta absoluta de usuario.
candidatos = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
ROOT = next(
    (p for p in candidatos if p.name == "AgroCebada-FIRA-2026"),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        "No se encontró la carpeta raíz 'AgroCebada-FIRA-2026'. "
        "Abre Jupyter dentro del proyecto o ajusta ROOT manualmente."
    )

MATRIX_PATH = ROOT / "notebooks" / "data" / "interim" / "tabular" / "exp38_c_matriz_maestra.csv"
FOLDS_PATH = ROOT / "notebooks" / "EXP24_predicciones_oof.csv"

OUT_METRICS = ROOT / "reports" / "metrics"
OUT_TABLES = ROOT / "reports" / "tables"
OUT_MODEL = ROOT / "models" / "reconstruccion_rf124_leaf16"

for path in [MATRIX_PATH, FOLDS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo requerido: {path}")

print("Raíz:", ROOT)
print("Matriz maestra:", MATRIX_PATH)
print("Folds originales:", FOLDS_PATH)

In [ ]:
# Cargar matriz maestra y archivo de folds originales.
matriz = pd.read_csv(MATRIX_PATH)
folds_originales = pd.read_csv(FOLDS_PATH)

ID = "ID_POLIGONO"
TARGET = "RENDIMIENTO_T_HA"
SPLIT = "CONJUNTO"
FOLD = "fold_espacial"

columnas_obligatorias = {ID, TARGET, SPLIT}
faltantes = sorted(columnas_obligatorias - set(matriz.columns))
if faltantes:
    raise ValueError(f"Faltan columnas obligatorias en la matriz: {faltantes}")

if matriz[ID].astype(str).duplicated().any():
    raise ValueError("La matriz maestra contiene IDs duplicados.")

matriz[ID] = matriz[ID].astype(str).str.strip()
matriz[TARGET] = pd.to_numeric(matriz[TARGET], errors="coerce")

n_etiquetadas = int(matriz[TARGET].notna().sum())
n_sin_etiqueta = int(matriz[TARGET].isna().sum())

assert len(matriz) == 197, f"Se esperaban 197 parcelas; se encontraron {len(matriz)}."
assert matriz[ID].nunique() == 197
assert n_etiquetadas == 138, f"Se esperaban 138 parcelas etiquetadas; hay {n_etiquetadas}."
assert n_sin_etiqueta == 59, f"Se esperaban 59 parcelas sin etiqueta; hay {n_sin_etiqueta}."

print("Dimensiones de la matriz:", matriz.shape)
print("Parcelas únicas:", matriz[ID].nunique())
print("Parcelas etiquetadas:", n_etiquetadas)
print("Parcelas para predecir:", n_sin_etiqueta)
print("Columnas totales:", len(matriz.columns))

In [ ]:
# Reproducir exactamente la selección histórica de RF124.
# Se conserva el orden original de selección para favorecer la reproducción
# determinista de los árboles y de las predicciones OOF.

excluir = {ID, TARGET, SPLIT}
predictoras = [c for c in matriz.columns if c not in excluir]

satelite_selector_historico = [
    c for c in predictoras
    if any(s in c.lower() for s in ["_inicial", "_intermedia", "_tardia"])
]
clima_selector_historico = [
    c for c in predictoras
    if any(s in c.lower() for s in [
        "precip_", "precip_mm_", "tmax_", "tmin_", "tmedia_"
    ])
]
area = ["AREA_HA"] if "AREA_HA" in predictoras else []

columnas = list(dict.fromkeys(
    satelite_selector_historico + clima_selector_historico + area
))

if len(columnas) != 124:
    raise ValueError(
        f"RF124 requiere 124 variables; la selección histórica produjo {len(columnas)}."
    )

# Comprobar las cuatro variables candidatas excluidas en EXP38-C.
topografia_excluida_esperada = {
    "elevation_coverage", "elevation_m", "slope_coverage", "slope_deg"
}
candidatas_128 = [c for c in matriz.columns if c not in excluir]
excluidas_rf124 = set(candidatas_128) - set(columnas)

if excluidas_rf124 != topografia_excluida_esperada:
    raise ValueError(
        "La selección no coincide con la reconstrucción histórica. "
        f"Variables excluidas observadas: {sorted(excluidas_rf124)}"
    )

# Clasificar por fuente para informar correctamente; no se infiere la fuente
# únicamente a partir de sufijos temporales.
features_satelite = [
    c for c in columnas
    if c.startswith(("ndvi_", "evi_", "lai_", "msavi_", "observaciones_"))
]
features_clima = [c for c in columnas if c not in features_satelite and c != "AREA_HA"]

assert len(features_satelite) == 51, f"Se esperaban 51 variables satelitales; hay {len(features_satelite)}."
assert len(features_clima) == 72, f"Se esperaban 72 variables climáticas; hay {len(features_clima)}."
assert area == ["AREA_HA"]
assert len(set(columnas)) == 124

print("Variables RF124:", len(columnas))
print("Variables satelitales:", len(features_satelite))
print("Variables climáticas:", len(features_clima))
print("Variable de superficie:", area)
print("Variables topográficas excluidas:", sorted(excluidas_rf124))

In [ ]:
# Separar datos etiquetados y parcelas que requieren predicción.
datos = matriz.loc[matriz[TARGET].notna()].copy()
sin_etiqueta = matriz.loc[matriz[TARGET].isna()].copy()

X = datos[columnas].apply(pd.to_numeric, errors="coerce")
X_pred = sin_etiqueta[columnas].apply(pd.to_numeric, errors="coerce")
y = datos[TARGET].to_numpy(dtype=float)

for nombre, bloque in [("entrenamiento", X), ("predicción", X_pred)]:
    valores = bloque.to_numpy(dtype=float)
    if not np.isfinite(valores).all():
        columnas_invalidas = bloque.columns[
            ~np.isfinite(valores).all(axis=0)
        ].tolist()
        raise ValueError(
            f"Hay valores faltantes o infinitos en {nombre}: {columnas_invalidas}"
        )

assert X.shape == (138, 124)
assert X_pred.shape == (59, 124)
assert len(y) == 138

print("X de entrenamiento:", X.shape)
print("y:", y.shape)
print("X para predicción:", X_pred.shape)
print("Valores faltantes/infinitos en X:", int((~np.isfinite(X.to_numpy(dtype=float))).sum()))
print("Valores faltantes/infinitos en X_pred:", int((~np.isfinite(X_pred.to_numpy(dtype=float))).sum()))

In [ ]:
# Recuperar la asignación espacial original de los folds.
columnas_fold = {ID, FOLD}
if not columnas_fold.issubset(folds_originales.columns):
    raise ValueError(
        f"El archivo de folds debe incluir {sorted(columnas_fold)}. "
        f"Columnas encontradas: {folds_originales.columns.tolist()}"
    )

fold_map = (
    folds_originales[[ID, FOLD]]
    .drop_duplicates()
    .copy()
)
fold_map[ID] = fold_map[ID].astype(str).str.strip()

if fold_map[ID].duplicated().any():
    raise ValueError("Hay parcelas asignadas a más de un fold espacial.")

if len(fold_map) != 138:
    raise ValueError(f"Se esperaban 138 IDs con fold; se encontraron {len(fold_map)}.")

datos = datos.drop(columns=[FOLD], errors="ignore").merge(
    fold_map, on=ID, how="left", validate="one_to_one"
)

if datos[FOLD].isna().any():
    faltantes_fold = datos.loc[datos[FOLD].isna(), ID].tolist()
    raise ValueError(f"Hay parcelas sin fold espacial: {faltantes_fold}")

fold_ids = datos[FOLD].to_numpy()
folds_unicos = sorted(pd.unique(fold_ids).tolist())

conteo_folds = datos.groupby(FOLD).size().rename("parcelas").reset_index()
display(conteo_folds)

assert folds_unicos == [1, 2, 3, 4, 5], f"Folds inesperados: {folds_unicos}"
assert conteo_folds["parcelas"].tolist() == [38, 25, 25, 25, 25]

print("Asignación espacial original recuperada:", len(datos), "parcelas")

## Validación cruzada espacial

Se preservan los cinco folds espaciales originales. En cada iteración, cuatro folds se utilizan para entrenar y el fold restante para evaluar. Las parcelas del fold evaluado no participan en el ajuste de ese modelo.

Se comparan `min_samples_leaf=8` y `min_samples_leaf=16` usando los mismos 124 predictores, la misma semilla y los mismos folds.

In [ ]:
# Evaluación OOF de RF124 leaf8 y RF124 leaf16.
configuraciones = {
    "RF124_leaf8": 8,
    "RF124_leaf16": 16,
}

resultados_globales = []
resultados_por_fold = []
predicciones_oof = []

for nombre_modelo, min_leaf in configuraciones.items():
    pred_oof = np.full(len(datos), np.nan, dtype=float)

    for fold in folds_unicos:
        mascara_test = fold_ids == fold
        mascara_train = ~mascara_test

        modelo_cv = RandomForestRegressor(
            n_estimators=500,
            min_samples_leaf=min_leaf,
            max_features=1.0,
            random_state=2026,
            n_jobs=-1,
        )
        modelo_cv.fit(X.loc[mascara_train], y[mascara_train])
        pred_oof[mascara_test] = modelo_cv.predict(X.loc[mascara_test])

        y_fold = y[mascara_test]
        p_fold = pred_oof[mascara_test]
        resultados_por_fold.append({
            "modelo": nombre_modelo,
            "fold_espacial": int(fold),
            "n": int(mascara_test.sum()),
            "RMSE": float(np.sqrt(mean_squared_error(y_fold, p_fold))),
            "MAE": float(mean_absolute_error(y_fold, p_fold)),
            "R2": float(r2_score(y_fold, p_fold)),
            "sesgo_real_menos_predicho": float(np.mean(y_fold - p_fold)),
        })

    if not np.isfinite(pred_oof).all():
        raise ValueError(f"{nombre_modelo}: faltan predicciones fuera de muestra.")

    resultados_globales.append({
        "modelo": nombre_modelo,
        "n_variables": len(columnas),
        "n_oof": len(y),
        "RMSE": float(np.sqrt(mean_squared_error(y, pred_oof))),
        "MAE": float(mean_absolute_error(y, pred_oof)),
        "R2": float(r2_score(y, pred_oof)),
        "sesgo_real_menos_predicho": float(np.mean(y - pred_oof)),
    })

    pred_df = datos[[ID, FOLD, TARGET]].copy()
    pred_df = pred_df.rename(columns={TARGET: "rendimiento_real"})
    pred_df["prediccion"] = pred_oof
    pred_df["modelo"] = nombre_modelo
    predicciones_oof.append(pred_df)

metricas_globales = pd.DataFrame(resultados_globales)
metricas_por_fold = pd.DataFrame(resultados_por_fold)
predicciones_oof = pd.concat(predicciones_oof, ignore_index=True)

print("Métricas globales OOF:")
display(metricas_globales.round(6))

print("Métricas por fold:")
display(metricas_por_fold.round(6))

In [ ]:
# Comparar con las métricas históricas documentadas para EXP38-J / RF124 leaf16.
historicas = {
    "RMSE": 0.516114,
    "MAE": 0.346521,
    "R2": 0.629638,
    "sesgo_real_menos_predicho": -0.026859,
}

fila_leaf16 = metricas_globales.loc[
    metricas_globales["modelo"].eq("RF124_leaf16")
].iloc[0]

comparacion_historica = pd.DataFrame({
    "metrica": list(historicas.keys()),
    "historico": list(historicas.values()),
    "recalculado": [fila_leaf16[k] for k in historicas],
})
comparacion_historica["diferencia_absoluta"] = (
    comparacion_historica["recalculado"] - comparacion_historica["historico"]
).abs()

display(comparacion_historica)

if not np.allclose(
    comparacion_historica["recalculado"].to_numpy(dtype=float),
    comparacion_historica["historico"].to_numpy(dtype=float),
    atol=1e-4,
    rtol=0,
):
    warnings.warn(
        "Las métricas no coinciden exactamente con el histórico. "
        "Revisa versión de scikit-learn, matriz, orden de variables y folds."
    )
else:
    print("Las métricas RF124 leaf16 coinciden con el histórico dentro de la tolerancia.")

In [ ]:
# Guardar evidencia de validación OOF.
OUT_METRICS.mkdir(parents=True, exist_ok=True)
OUT_TABLES.mkdir(parents=True, exist_ok=True)

metricas_globales.to_csv(
    OUT_METRICS / "rf124_leaf16_comparacion_oof.csv",
    index=False,
    encoding="utf-8-sig",
)
metricas_por_fold.to_csv(
    OUT_TABLES / "rf124_leaf16_metricas_por_fold.csv",
    index=False,
    encoding="utf-8-sig",
)
predicciones_oof.to_csv(
    OUT_TABLES / "rf124_leaf16_predicciones_oof.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Evidencia OOF guardada en:")
print(OUT_METRICS / "rf124_leaf16_comparacion_oof.csv")
print(OUT_TABLES / "rf124_leaf16_metricas_por_fold.csv")
print(OUT_TABLES / "rf124_leaf16_predicciones_oof.csv")

## Entrenamiento final y predicción

Después de evaluar las configuraciones con validación espacial, se ajusta RF124 leaf16 con las 138 parcelas etiquetadas. Este modelo final se utiliza para estimar el rendimiento de las 59 parcelas sin etiqueta. Las predicciones de estas 59 parcelas no se usan para calcular las métricas de validación.

In [ ]:
# Entrenar el modelo final con todas las parcelas etiquetadas.
modelo_final = RandomForestRegressor(
    n_estimators=500,
    min_samples_leaf=16,
    max_features=1.0,
    random_state=2026,
    n_jobs=-1,
)
modelo_final.fit(X, y)

pred_final = modelo_final.predict(X_pred)

if len(pred_final) != 59 or not np.isfinite(pred_final).all():
    raise ValueError("Las predicciones finales no son completas o contienen valores no finitos.")

resultado_predicciones = sin_etiqueta[[ID]].copy()
resultado_predicciones["RENDIMIENTO_PREDICHO_T_HA"] = pred_final

print("Modelo final entrenado.")
print("Parcelas usadas para el ajuste final:", len(X))
print("Variables:", X.shape[1])
print("Parcelas predichas:", len(resultado_predicciones))
display(resultado_predicciones.head(10))
display(resultado_predicciones["RENDIMIENTO_PREDICHO_T_HA"].describe().round(4))

In [ ]:
# Guardar modelo, predicciones y metadatos de reconstrucción.
import joblib

OUT_MODEL.mkdir(parents=True, exist_ok=True)

modelo_path = OUT_MODEL / "RF124_leaf16_final.joblib"
pred_path = OUT_MODEL / "RF124_leaf16_predicciones_59.csv"
metadata_path = OUT_MODEL / "metadata.json"

joblib.dump(modelo_final, modelo_path)
resultado_predicciones.to_csv(pred_path, index=False, encoding="utf-8-sig")

metadata = {
    "modelo": "RF124_leaf16",
    "matriz_origen": str(MATRIX_PATH.relative_to(ROOT)),
    "folds_origen": str(FOLDS_PATH.relative_to(ROOT)),
    "n_parcelas_matriz": int(len(matriz)),
    "n_entrenamiento_final": int(len(X)),
    "n_prediccion": int(len(X_pred)),
    "n_variables": int(len(columnas)),
    "variables": columnas,
    "n_estimators": 500,
    "min_samples_leaf": 16,
    "max_features": 1.0,
    "random_state": 2026,
    "sklearn_version": sklearn.__version__,
    "metricas_oof_leaf16": {
        "RMSE": float(fila_leaf16["RMSE"]),
        "MAE": float(fila_leaf16["MAE"]),
        "R2": float(fila_leaf16["R2"]),
        "sesgo_real_menos_predicho": float(fila_leaf16["sesgo_real_menos_predicho"]),
    },
    "nota": (
        "El modelo final se ajustó con las 138 parcelas etiquetadas. "
        "Las métricas provienen de validación espacial OOF, no del ajuste final."
    ),
}
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Modelo:", modelo_path)
print("Predicciones:", pred_path)
print("Metadatos:", metadata_path)

In [ ]:
# Prueba de integridad de los artefactos guardados.
modelo_recargado = joblib.load(modelo_path)
predicciones_recargadas = pd.read_csv(pred_path)

pred_recargadas = modelo_recargado.predict(X_pred)

assert len(modelo_recargado.feature_names_in_) == 124
assert list(modelo_recargado.feature_names_in_) == list(columnas)
assert len(predicciones_recargadas) == 59
assert predicciones_recargadas[ID].is_unique
assert predicciones_recargadas["RENDIMIENTO_PREDICHO_T_HA"].notna().all()
assert np.allclose(
    pred_recargadas,
    predicciones_recargadas["RENDIMIENTO_PREDICHO_T_HA"].to_numpy(),
    atol=1e-10,
    rtol=0,
)

print("Prueba de recarga y reproducción de predicciones: OK")
print("Métricas OOF de RF124 leaf16:")
print(
    metricas_globales.loc[
        metricas_globales["modelo"].eq("RF124_leaf16"),
        ["RMSE", "MAE", "R2", "sesgo_real_menos_predicho"]
    ].round(6).to_string(index=False)
)

## Interpretación final

La validación histórica de RF124 leaf16 reportó RMSE ≈ 0.5161 t/ha, MAE ≈ 0.3465 t/ha, R² ≈ 0.6296 y sesgo medio real menos predicho ≈ −0.0269 t/ha. Estos valores se consideran reproducidos solo si la comparación de la celda anterior coincide dentro de la tolerancia indicada.

El archivo de predicciones final contiene estimaciones para las 59 parcelas sin rendimiento observado. No deben interpretarse como mediciones reales ni utilizarse para calcular métricas de desempeño.